# Gradient boosting — lbw cohort

Train **gradient boosting** on the `lbw` cohort and tune it with cluster-grouped
cross-validation on the training rows, then score the held-out test split once (section 4).

Owner: Jeff  ·  **`random_state=42` so results reproduce.**

## 1 — Load the data

In [ ]:
import sys
from pathlib import Path

# make the repo root importable when run from ml/notebooks/modeling
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ml" / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ml.src.dataset import build_cohort, cv_folds, SELECTED, OUTCOMES

COHORT = "lbw"
# load the combined, cleaned cohort
df = build_cohort(COHORT)

# training rows for tuning; test rows are used once, in section 4
train = df[df["split"] == "train"].reset_index(drop=True)
print(f"train rows: {len(train):,}  clusters: {train['cluster_id'].nunique():,}")

## 2 — Features and target
Features are the cohort's `SELECTED` columns minus the outcome and the design columns.

In [ ]:
TARGET = OUTCOMES[COHORT]
DESIGN = ["split", "cluster_id", "country", "weight"]

# features = SELECTED minus outcome and design columns
FEATURES = [c for c in SELECTED[COHORT] if c not in DESIGN + [TARGET]]
X_train, y_train = train[FEATURES], train[TARGET]

# event count and rate (the constant-baseline PR-AUC)
print(f"{len(FEATURES)} features: {FEATURES}")
print(f"events: {int(y_train.sum()):,}  rate: {y_train.mean():.2%}")
# blanks per feature (first pregnancies have no interval by design)
print(X_train.isna().sum()[lambda s: s > 0].to_dict())

## 3 — Train and tune
`HistGradientBoostingClassifier`: supports `class_weight='balanced'` for the rare outcome
(no resampling) and handles blanks natively, so first-pregnancy intervals stay blank.
Tuned with `GridSearchCV` on `cv_folds()` (5 cluster-grouped folds), PR-AUC first.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

# balanced class weights for the imbalance; no early stopping so max_iter is tuned
gb = HistGradientBoostingClassifier(class_weight="balanced", early_stopping=False, random_state=42)
# education is ordered, so encode it as 0-3 and let the trees split on order
EDU_ORDER = ["No education", "Primary", "Secondary", "Higher"]
prep = ColumnTransformer(
    [("edu", OrdinalEncoder(categories=[EDU_ORDER]), ["education"])],
    remainder="passthrough", verbose_feature_names_out=False,
)
pipe = Pipeline([("prep", prep), ("model", gb)])

# small grid of simple models: 48 settings x 5 folds
GRID = {
    "model__learning_rate": [0.02, 0.05],
    "model__max_iter": [50, 100],
    "model__max_leaf_nodes": [4, 7],
    "model__min_samples_leaf": [20, 80, 150],
    "model__l2_regularization": [1.0, 5.0],
}

# cluster-grouped folds from dataset.py
folds = cv_folds(train, COHORT)
search = GridSearchCV(pipe, GRID, cv=folds, scoring=["average_precision", "roc_auc"],
                      refit="average_precision", n_jobs=1)
# fit on training rows only
search.fit(X_train, y_train)

In [ ]:
# per-fold scores of the best setting
res, k = search.cv_results_, search.best_index_
fold_scores = pd.DataFrame({
    "pr_auc": [res[f"split{i}_test_average_precision"][k] for i in range(len(folds))],
    "roc_auc": [res[f"split{i}_test_roc_auc"][k] for i in range(len(folds))],
    # constant baseline: PR-AUC of a flat score = event rate in the fold
    "baseline_pr_auc": [y_train.iloc[val].mean() for _, val in folds],
}, index=pd.Index(range(1, len(folds) + 1), name="fold"))

# mean and sd over folds
summary = fold_scores.agg(["mean", "std"]).T.round(4)
print(summary)
print("best params:", {p.replace("model__", ""): v for p, v in search.best_params_.items()})

### CV results (training rows only)

- PR-AUC **0.199 ± 0.017** over 5 cluster-grouped folds, against a constant baseline of 0.082 (about 2.4×).
- ROC-AUC **0.614 ± 0.028**.
- Best setting: learning_rate 0.05, max_iter 100, max_leaf_nodes 7, min_samples_leaf 20, l2_regularization 5.
- A wider grid (up to 300 trees, 15 leaves) gave the same PR-AUC (0.199), so more complexity adds nothing.

In [ ]:
# fold scores against the constant baseline
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(fold_scores.index, fold_scores["pr_auc"], label="gradient boosting")
axes[0].plot(fold_scores.index, fold_scores["baseline_pr_auc"], "k--o", label="constant baseline")
axes[0].set(title="CV PR-AUC by fold", xlabel="fold", ylabel="PR-AUC")
axes[0].legend()
axes[1].bar(fold_scores.index, fold_scores["roc_auc"])
axes[1].axhline(0.5, color="k", ls="--")
axes[1].set(title="CV ROC-AUC by fold", xlabel="fold", ylabel="ROC-AUC", ylim=(0.4, 1))
fig.suptitle(f"{COHORT}: gradient boosting, best grid setting")
fig.tight_layout()
plt.show()

## 4 — Evaluate on the test split

The best CV setting, already refit on all training rows by `GridSearchCV`, scores the held-out
test clusters once. Nothing is retuned after this. PR-AUC is the headline; the checklist comparison
flags the same share of test pregnancies as the current high-risk checklist.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score

# test rows: the held-out clusters, used once
test = df[df["split"] == "test"].reset_index(drop=True)
X_test, y_test = test[FEATURES], test[TARGET]
print(f"test rows: {len(test):,}  events: {int(y_test.sum()):,}  rate: {y_test.mean():.2%}")

# risk scores from the best setting (refit on all training rows)
model = search.best_estimator_
score = model.predict_proba(X_test)[:, 1]

In [ ]:
# test PR-AUC and ROC-AUC, pooled and per country (country is not a feature)
def metrics(mask):
    y, s = y_test[mask], score[mask]
    return {"rows": int(mask.sum()), "events": int(y.sum()), "baseline_pr_auc": y.mean(),
            "pr_auc": average_precision_score(y, s), "roc_auc": roc_auc_score(y, s)}


test_scores = pd.DataFrame({
    "pooled": metrics(np.ones(len(test), bool)),
    "Kenya": metrics((test["country"] == "Kenya").to_numpy()),
    "Tanzania": metrics((test["country"] == "Tanzania").to_numpy()),
}).T
print(test_scores.round(3))

# CV figures for comparison
print(summary.loc[["pr_auc", "roc_auc"]])

In [ ]:
# checklist flag: same rule as checklist_flag in dataset.py (not in SELECTED, so rebuilt here)
checklist = (
    (test["maternal_age"] < 18) | (test["maternal_age"] >= 35)
    | (test["prior_live_births"] >= 5) | (test["preceding_interval_m"] < 24)
    | (test["multiple"] == 1) | (test["prior_stillbirth"] == 1)
).to_numpy()
n_flag = int(checklist.sum())

# model flags the same number of pregnancies: the highest scores
model_flag = np.zeros(len(test), bool)
model_flag[np.argsort(-score, kind="stable")[:n_flag]] = True

# events caught, sensitivity and PPV at the same flag rate
events = y_test.to_numpy() == 1
compare = pd.DataFrame({
    "flagged": [n_flag, n_flag],
    "events_caught": [int((checklist & events).sum()), int((model_flag & events).sum())],
}, index=["checklist", "gradient boosting"])
compare["sensitivity"] = compare["events_caught"] / events.sum()
compare["ppv"] = compare["events_caught"] / compare["flagged"]
print(f"flag rate: {checklist.mean():.1%} of test pregnancies ({n_flag:,})")
print(compare.round(3))

# many rows share a score at the cut-off: range of events the model could catch
cut = np.sort(score)[::-1][n_flag - 1]
above, tied = score > cut, score == cut
slots = n_flag - int(above.sum())
low = int((above & events).sum()) + max(0, slots - int((tied & ~events).sum()))
high = int((above & events).sum()) + min(slots, int((tied & events).sum()))
print(f"{int(tied.sum())} rows tied at the cut-off; model catches {low}-{high} events")

In [ ]:
# calibration: mean predicted vs observed, and 10 equal-width bins
print(f"mean predicted {score.mean():.3f}  observed rate {y_test.mean():.3f}")
obs, pred = calibration_curve(y_test, score, n_bins=10)

# PR curve with the constant baseline and the checklist point; reliability plot
prec, rec, _ = precision_recall_curve(y_test, score)
pooled_pr = test_scores.loc["pooled", "pr_auc"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(rec, prec, label=f"gradient boosting (PR-AUC {pooled_pr:.3f})")
axes[0].axhline(y_test.mean(), color="k", ls="--", label=f"constant baseline ({y_test.mean():.3f})")
axes[0].plot(compare.loc["checklist", "sensitivity"], compare.loc["checklist", "ppv"], "rs",
             label="checklist")
axes[0].set(title="Test precision-recall", xlabel="recall (sensitivity)", ylabel="precision (PPV)")
axes[0].legend()
axes[1].plot(pred, obs, "o-", label="gradient boosting")
axes[1].plot([0, 1], [0, 1], "k--", label="perfect calibration")
axes[1].set(title="Test reliability (10 bins)", xlabel="mean predicted", ylabel="observed rate")
axes[1].legend()
fig.suptitle(f"{COHORT}: gradient boosting on the test split")
fig.tight_layout()
plt.show()

### Test results

- **PR-AUC 0.159** on 1,849 test births (130 LBW), against a constant baseline of 0.070: about 2.3× better than flagging at random.
- Lower than CV (0.199 ± 0.017), but the test event rate is lower too (7.0% vs 8.2% in training). The gain over the baseline is close to CV (2.3× vs 2.4×), and ROC-AUC held: 0.618 vs 0.614 ± 0.028.
- Tanzania is stronger than Kenya: PR-AUC 0.228 vs 0.117 (baselines 0.067 and 0.074), ROC-AUC 0.637 vs 0.593. This is the opposite of the ANC cohort.
- The checklist flags 41.6% of test births (770) and catches 69 of 130 LBW (sensitivity 0.53, PPV 0.090).
- Flagging the same 770 by model score catches 73 (sensitivity 0.56, PPV 0.095): 4 more (72–74 with tied scores). That is close to the checklist, not a clear win.
- Probabilities are not risks: mean predicted 0.45 against an observed rate of 0.070, because `class_weight="balanced"` inflates them. Use the score to rank only. The top bins hold few births, so the reliability plot is noisy there.
- For the final model: gradient boosting ranks LBW better than chance but adds little over the checklist at its flag rate. Compare with LR/RF on test PR-AUC before choosing.

## 5 — Save the result

Values for `db.metrics.save_metrics(...)` (one row in `model_metrics`):

| cohort | algorithm | pr_auc | roc_auc | recall_at_flag | beats_checklist | test_events |
|---|---|---|---|---|---|---|
| lbw | gradient_boosting | 0.159 | 0.618 | 0.562 | True (by 4 events) | 130 |

`recall_at_flag` is the model's sensitivity at the checklist's flag rate (41.6% of test rows).
The database save runs once the app Postgres is up.